# 02 - Data Preprocessing

This notebook prepares the supply chain dataset for analysis.

The preprocessing pipeline includes:

- Missing-value inspection and imputation
- Outlier treatment
- Numeric normalization
- Categorical encoding
- Binning of continuous variables

The original raw dataset is not overwritten. New transformed columns are created
so that the effects of preprocessing can be compared with the original values.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("CS675SupplyChainPreprocessing")
    .getOrCreate()
)

inventory_path = "/home/jovyan/work/CS675_Supply_Chain_Project/data/raw/supply_chain_dataset1.csv"

inventory_df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv(inventory_path)
)

print("Rows:", inventory_df.count())
print("Columns:", len(inventory_df.columns))

Rows: 91250
Columns: 15


In [2]:
# Count null values in every column before preprocessing.

missing_values_df = inventory_df.select(
    [
        F.sum(
            F.col(column_name).isNull().cast("int")
        ).alias(column_name)
        for column_name in inventory_df.columns
    ]
)

missing_values_df.show(truncate=False)

+----+------+------------+-----------+------+----------+---------------+-----------------------+-------------+--------------+---------+----------+--------------+-------------+---------------+
|Date|SKU_ID|Warehouse_ID|Supplier_ID|Region|Units_Sold|Inventory_Level|Supplier_Lead_Time_Days|Reorder_Point|Order_Quantity|Unit_Cost|Unit_Price|Promotion_Flag|Stockout_Flag|Demand_Forecast|
+----+------+------------+-----------+------+----------+---------------+-----------------------+-------------+--------------+---------+----------+--------------+-------------+---------------+
|0   |0     |0           |0          |0     |0         |0              |0                      |0            |0             |0        |0         |0             |0            |0              |
+----+------+------------+-----------+------+----------+---------------+-----------------------+-------------+--------------+---------+----------+--------------+-------------+---------------+



In [3]:
from pyspark.ml.feature import Imputer

In [4]:
impute_columns = [
    "Units_Sold",
    "Inventory_Level",
    "Supplier_Lead_Time_Days",
    "Reorder_Point",
    "Unit_Cost",
    "Unit_Price",
    "Demand_Forecast"
]

In [5]:
imputed_columns = [
    f"{column}_Imputed"
    for column in impute_columns
]

In [6]:
# Median imputation is less sensitive to extreme values than mean imputation.

imputer = Imputer(
    inputCols=impute_columns,
    outputCols=imputed_columns,
    strategy="median"
)

In [7]:
imputer_model = imputer.fit(inventory_df)

preprocessed_df = imputer_model.transform(inventory_df)

In [8]:
preprocessed_df.select(
    "Units_Sold",
    "Units_Sold_Imputed",
    "Inventory_Level",
    "Inventory_Level_Imputed"
).show(10)

+----------+------------------+---------------+-----------------------+
|Units_Sold|Units_Sold_Imputed|Inventory_Level|Inventory_Level_Imputed|
+----------+------------------+---------------+-----------------------+
|        10|                10|            592|                    592|
|        17|                17|            575|                    575|
|        35|                35|            540|                    540|
|        24|                24|            516|                    516|
|        21|                21|            495|                    495|
|        18|                18|            477|                    477|
|        19|                19|            458|                    458|
|        23|                23|            435|                    435|
|        25|                25|            410|                    410|
|        25|                25|            385|                    385|
+----------+------------------+---------------+-----------------

In [9]:
# Confirm that no values changed because the original dataset had no nulls.

difference_count = preprocessed_df.filter(
    F.col("Units_Sold") != F.col("Units_Sold_Imputed")
).count()

print("Rows changed by Units_Sold imputation:", difference_count)

Rows changed by Units_Sold imputation: 0


In [10]:
# Recalculate the IQR boundaries for Units_Sold.
# We use these boundaries to create a capped version of the variable
# without deleting potentially valid high-demand records.

q1, q3 = preprocessed_df.approxQuantile(
    "Units_Sold_Imputed",
    [0.25, 0.75],
    0.01
)

iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

print("Q1:", q1)
print("Q3:", q3)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)

Q1: 13.0
Q3: 27.0
Lower bound: -8.0
Upper bound: 48.0


In [11]:
# Create a capped version of Units_Sold.
# Values above the upper IQR boundary are capped at the boundary.
# Values below the lower boundary would also be capped, although none are expected here.

preprocessed_df = preprocessed_df.withColumn(
    "Units_Sold_Capped",
    F.when(
        F.col("Units_Sold_Imputed") > upper_bound,
        F.lit(upper_bound)
    )
    .when(
        F.col("Units_Sold_Imputed") < lower_bound,
        F.lit(lower_bound)
    )
    .otherwise(F.col("Units_Sold_Imputed"))
)

In [12]:
# Show only records where capping changed the original value.

preprocessed_df.filter(
    F.col("Units_Sold_Imputed") != F.col("Units_Sold_Capped")
).select(
    "Date",
    "SKU_ID",
    "Warehouse_ID",
    "Units_Sold_Imputed",
    "Units_Sold_Capped",
    "Promotion_Flag",
    "Demand_Forecast"
).orderBy(
    F.desc("Units_Sold_Imputed")
).show(20, truncate=False)

+----------+------+------------+------------------+-----------------+--------------+---------------+
|Date      |SKU_ID|Warehouse_ID|Units_Sold_Imputed|Units_Sold_Capped|Promotion_Flag|Demand_Forecast|
+----------+------+------------+------------------+-----------------+--------------+---------------+
|2024-04-29|SKU_34|WH_4        |59                |48.0             |1             |57.98          |
|2024-03-26|SKU_7 |WH_4        |58                |48.0             |1             |59.43          |
|2024-04-23|SKU_38|WH_3        |58                |48.0             |1             |56.12          |
|2024-04-14|SKU_36|WH_3        |57                |48.0             |1             |54.67          |
|2024-04-12|SKU_3 |WH_1        |55                |48.0             |1             |54.19          |
|2024-04-13|SKU_33|WH_4        |55                |48.0             |1             |58.37          |
|2024-05-04|SKU_23|WH_3        |55                |48.0             |1             |54.19  

In [13]:
capped_count = preprocessed_df.filter(
    F.col("Units_Sold_Imputed") != F.col("Units_Sold_Capped")
).count()

print("Rows affected by Units_Sold capping:", capped_count)

Rows affected by Units_Sold capping: 110


In [14]:
effective_lower_bound = max(0, lower_bound)

print("Effective lower bound:", effective_lower_bound)

Effective lower bound: 0


In [15]:
preprocessed_df = preprocessed_df.withColumn(
    "Units_Sold_Capped",
    F.when(
        F.col("Units_Sold_Imputed") > upper_bound,
        F.lit(upper_bound)
    )
    .when(
        F.col("Units_Sold_Imputed") < effective_lower_bound,
        F.lit(effective_lower_bound)
    )
    .otherwise(F.col("Units_Sold_Imputed"))
)

In [16]:
# VectorAssembler combines multiple numeric columns into one feature vector.
# StandardScaler standardizes those features so they are on comparable scales.
from pyspark.ml.feature import VectorAssembler, StandardScaler

In [17]:
numeric_features = [
    "Units_Sold_Capped",
    "Inventory_Level_Imputed",
    "Supplier_Lead_Time_Days_Imputed",
    "Reorder_Point_Imputed",
    "Unit_Cost_Imputed",
    "Unit_Price_Imputed",
    "Demand_Forecast_Imputed"
]

In [18]:
assembler = VectorAssembler(
    inputCols=numeric_features,
    outputCol="numeric_features_vector"
)

preprocessed_df = assembler.transform(preprocessed_df)

In [19]:
preprocessed_df.select(
    "numeric_features_vector"
).show(5, truncate=False)

+-----------------------------------------+
|numeric_features_vector                  |
+-----------------------------------------+
|[10.0,592.0,14.0,379.0,13.95,20.48,8.52] |
|[17.0,575.0,14.0,379.0,13.95,20.48,18.63]|
|[35.0,540.0,14.0,379.0,13.95,20.48,39.62]|
|[24.0,516.0,14.0,379.0,13.95,20.48,19.43]|
|[21.0,495.0,14.0,379.0,13.95,20.48,18.7] |
+-----------------------------------------+
only showing top 5 rows


In [22]:
scaler = StandardScaler(
    inputCol="numeric_features_vector",
    outputCol="scaled_numeric_features",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(preprocessed_df)

preprocessed_df = scaler_model.transform(preprocessed_df)

IllegalArgumentException: requirement failed: Output column scaled_numeric_features already exists.

In [21]:
preprocessed_df.select(
    "numeric_features_vector",
    "scaled_numeric_features"
).show(5, truncate=False)

+-----------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|numeric_features_vector                  |scaled_numeric_features                                                                                                                   |
+-----------------------------------------+------------------------------------------------------------------------------------------------------------------------------------------+
|[10.0,592.0,14.0,379.0,13.95,20.48,8.52] |[-1.1097422737333755,0.9025357034816448,1.5394341985471003,1.4382667430174367,0.3817894724922661,0.3114952330249969,-1.2165496570829752]  |
|[17.0,575.0,14.0,379.0,13.95,20.48,18.63]|[-0.3368763676538002,0.7751834338974443,1.5394341985471003,1.4382667430174367,0.3817894724922661,0.3114952330249969,-0.1527819948970388]  |
|[35.0,540.0,14.0,379.0,13.95,20.48,39.62]|[1.650493105122251,0.5129875847535017,1.53

In [23]:
# StringIndexer converts text categories into numeric category indexes.
# OneHotEncoder then converts those indexes into binary indicator vectors.

from pyspark.ml.feature import StringIndexer, OneHotEncoder

In [24]:
# Convert Region values such as East, North, South, and West
# into numeric category indexes.

region_indexer = StringIndexer(
    inputCol="Region",
    outputCol="Region_Index"
)

region_indexer_model = region_indexer.fit(preprocessed_df)

preprocessed_df = region_indexer_model.transform(preprocessed_df)

In [25]:
preprocessed_df.select(
    "Region",
    "Region_Index"
).distinct().orderBy("Region_Index").show()

+------+------------+
|Region|Region_Index|
+------+------------+
| North|         0.0|
|  East|         1.0|
| South|         2.0|
|  West|         3.0|
+------+------------+



In [26]:
# Convert Region_Index into a one-hot encoded vector.
# This avoids treating the category numbers as if they had numeric order.

region_encoder = OneHotEncoder(
    inputCols=["Region_Index"],
    outputCols=["Region_Encoded"]
)

region_encoder_model = region_encoder.fit(preprocessed_df)

preprocessed_df = region_encoder_model.transform(preprocessed_df)

In [29]:
preprocessed_df.select(
    "Region",
    "Region_Index"
).distinct().orderBy("Region_Index").show()

+------+------------+
|Region|Region_Index|
+------+------------+
| North|         0.0|
|  East|         1.0|
| South|         2.0|
|  West|         3.0|
+------+------------+



In [31]:
preprocessed_df.select(
    "Region",
    "Region_Index",
    "Region_Encoded"
).distinct().orderBy("Region_Index").show(truncate=False)

+------+------------+--------------+
|Region|Region_Index|Region_Encoded|
+------+------------+--------------+
|North |0.0         |(3,[0],[1.0]) |
|East  |1.0         |(3,[1],[1.0]) |
|South |2.0         |(3,[2],[1.0]) |
|West  |3.0         |(3,[],[])     |
+------+------------+--------------+



In [32]:
from pyspark.ml.feature import Bucketizer

In [33]:
# Create demand bins using business-friendly sales ranges.
# 0-15   -> Low demand
# 16-30  -> Medium demand
# 31+    -> High demand

splits = [
    float("-inf"),
    16,
    31,
    float("inf")
]

In [34]:
demand_bucketizer = Bucketizer(
    splits=splits,
    inputCol="Units_Sold_Capped",
    outputCol="Demand_Bin"
)

preprocessed_df = demand_bucketizer.transform(preprocessed_df)

In [35]:
preprocessed_df = preprocessed_df.withColumn(
    "Demand_Category",
    F.when(F.col("Demand_Bin") == 0.0, "Low")
     .when(F.col("Demand_Bin") == 1.0, "Medium")
     .when(F.col("Demand_Bin") == 2.0, "High")
)

In [36]:
preprocessed_df.select(
    "Units_Sold",
    "Units_Sold_Capped",
    "Demand_Bin",
    "Demand_Category"
).show(20)

+----------+-----------------+----------+---------------+
|Units_Sold|Units_Sold_Capped|Demand_Bin|Demand_Category|
+----------+-----------------+----------+---------------+
|        10|             10.0|       0.0|            Low|
|        17|             17.0|       1.0|         Medium|
|        35|             35.0|       2.0|           High|
|        24|             24.0|       1.0|         Medium|
|        21|             21.0|       1.0|         Medium|
|        18|             18.0|       1.0|         Medium|
|        19|             19.0|       1.0|         Medium|
|        23|             23.0|       1.0|         Medium|
|        25|             25.0|       1.0|         Medium|
|        25|             25.0|       1.0|         Medium|
|        23|             23.0|       1.0|         Medium|
|        29|             29.0|       1.0|         Medium|
|        25|             25.0|       1.0|         Medium|
|        26|             26.0|       1.0|         Medium|
|        19|  

In [37]:
preprocessed_df.groupBy(
    "Demand_Category"
).count().orderBy(
    "Demand_Category"
).show()

+---------------+-----+
|Demand_Category|count|
+---------------+-----+
|           High|12203|
|            Low|30926|
|         Medium|48121|
+---------------+-----+



## Preprocessing Summary

The preprocessing pipeline included the required steps of missing-value
assessment, imputation, outlier treatment, normalization, categorical
encoding, and binning.

No missing values were present in the source dataset. Median imputation was
still implemented for selected numeric fields so that the preprocessing
pipeline would remain reusable if missing values are introduced in future
data.

Potential outliers in Units_Sold were identified using the IQR method.
Because the high-demand observations were often associated with promotions
and high demand forecasts, they were preserved as valid business events.
A capped Units_Sold feature was created for preprocessing rather than deleting
the original observations.

Selected numeric variables were standardized using StandardScaler so that
variables measured on different scales could be represented comparably.

Region was converted from a text category into indexed and one-hot encoded
features. One-hot encoding prevents arbitrary numeric category indexes from
being interpreted as ordered quantities.

Finally, Units_Sold_Capped was divided into Low, Medium, and High demand
categories. These business-friendly bins provide an interpretable way to
compare inventory behavior across different demand levels.

The raw columns remain available alongside the transformed features so that
original business values can still be used for descriptive analysis.